# Day6：集中训练、分散执行的数据流

教师提供的分段练习，供 RedMoon 阅读、修改与复跑。
本例演示两个局部 Actor 和一个集中式状态价值 Critic：
**训练时 Critic 可以使用联合信息，执行时每个 Actor 只使用自己的观测。**

只用三条手工教学经历执行一次 Actor–Critic 更新，不是完整 MAPPO，
不包含环境采样循环、PPO clip 或 GAE，不证明合作策略已经收敛。
网络初值也是为了读懂数值手工设置的，不是先训练好的答案。

## 1. 导入工具

`torch` 处理张量和梯度，`nn.Linear` 建立最小网络，
`Categorical` 把动作得分变成动作分布。本例不需要 GPU 或额外环境库。

In [1]:
import math
import torch
from torch import nn
from torch.distributions import Categorical

torch.manual_seed(2026)
torch.set_printoptions(precision=4, sci_mode=False)
print("PyTorch:", torch.__version__)

PyTorch: 2.10.0+cu128


## 2. 三条教学经历：一行就是同一次联合交互

`obs_0`、`obs_1` 分别保存两人的局部观测，每条观测有两个模拟特征。
两个特征只是用于演示张量接口，没有赋予真实物理意义。
第一行不能和第二行的队友观测随意拼接。

`actions_0`、`actions_1` 是已经执行过的动作，0 表示 A，1 表示 B。
两人选相同动作，团队奖励为 2；不同则为 0。
同一份团队奖励用于两个角色的学习，不再加成奖励 4。
三条经历都在这一步真正结束，所以完整回报就是当前奖励。

In [2]:
# 形状 (3, 2)：3 条经历，每条局部观测包含 2 个特征。
obs_0 = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])
obs_1 = torch.tensor([[0., 1.], [1., 0.], [0., 1.]])

# 每个位置都对应上面同一行经历，不能重新采一个动作替换它。
actions_0 = torch.tensor([0, 1, 0], dtype=torch.long)
actions_1 = torch.tensor([0, 1, 1], dtype=torch.long)

# 团队奖励 R，形状 (3,)；不是两份个体奖励相加。
rewards = torch.where(actions_0 == actions_1, 2.0, 0.0)
print("联合动作:", torch.stack([actions_0, actions_1], dim=-1))
print("团队奖励:", rewards)

联合动作: tensor([[0, 0],
        [1, 1],
        [0, 1]])
团队奖励: tensor([2., 2., 0.])


## 3. 三个分开的最小网络

每个 Actor 的输入是自己的 2 个特征，输出 A/B 的 2 个得分。
Critic 输入两份观测拼成的 4 个特征，输出一个团队回报预测。
本例三个网络不共享参数，便于看清梯度各走哪条路。
参数共享也是另一种合法设计，不是 CTDE 的必要条件。

Actor 权重全为 0，让初始动作概率恰好为 0.5/0.5。
Critic 权重全为 0.1，让它最初输出四个特征之和的 0.1 倍。
这个 0.1 是教学初值，不是准确价值。

In [3]:
actor_0 = nn.Linear(2, 2, bias=False)
actor_1 = nn.Linear(2, 2, bias=False)
critic = nn.Linear(4, 1, bias=False)

with torch.no_grad():
    actor_0.weight.zero_()
    actor_1.weight.zero_()
    critic.weight.fill_(0.1)

parameters = (
    list(actor_0.parameters())
    + list(actor_1.parameters())
    + list(critic.parameters())
)
optimizer = torch.optim.SGD(parameters, lr=0.05)

## 4. 前向计算：谁能看到什么

`torch.cat(..., dim=-1)` 沿特征列拼接，同一行仍是一条经历：
`(3, 2)` 与 `(3, 2)` 变成 `(3, 4)`。
`joint_features` 是本例的联合观测特征，不能一般性地说它包含全部全局状态。

两个 Actor 仍分别只输入 `obs_0`、`obs_1`。
`value` 是状态价值预测，不是选动作的指令，也没偷看本条已经执行的动作。
`squeeze(-1)` 只去掉 Critic 输出末尾的单值维度，保留批次维度。

In [4]:
joint_features = torch.cat([obs_0, obs_1], dim=-1)
dist_0 = Categorical(logits=actor_0(obs_0))
dist_1 = Categorical(logits=actor_1(obs_1))
value = critic(joint_features).squeeze(-1)

print("联合特征形状:", tuple(joint_features.shape))
print("Actor 0 的 A/B 概率:\n", dist_0.probs)
print("Actor 1 的 A/B 概率:\n", dist_1.probs)
print("Critic 预测:", value.detach())

联合特征形状: (3, 4)
Actor 0 的 A/B 概率:
 tensor([[0.5000, 0.5000],
        [0.5000, 0.5000],
        [0.5000, 0.5000]], grad_fn=<SoftmaxBackward0>)
Actor 1 的 A/B 概率:
 tensor([[0.5000, 0.5000],
        [0.5000, 0.5000],
        [0.5000, 0.5000]], grad_fn=<SoftmaxBackward0>)
Critic 预测: tensor([0.2000, 0.2000, 0.3000])


## 5. 固定目标与评分

本例是真正终止的单步经历，Critic 的目标 `targets` 就是团队奖励。
保存更新前的 `old_values`，用 `targets - old_values` 得到本次固定评分。
两个角色在本例中收到相同的团队奖励、使用同一个团队价值参照，
所以使用同一组评分；这不意味着已经分清谁对奖励贡献了多少。

`detach()` 阻止 Actor loss 沿评分回传到 Critic。
更新 Critic 时仍使用保留计算图的 `value`，不能把它也 detach。

In [5]:
targets = rewards.detach().clone()
old_values = value.detach().clone()
advantages = (targets - old_values).detach()

print("固定 Critic 目标:", targets)
print("更新前价值快照:", old_values)
print("固定 Actor 评分:", advantages)

固定 Critic 目标: tensor([2., 2., 0.])
更新前价值快照: tensor([0.2000, 0.2000, 0.3000])
固定 Actor 评分: tensor([ 1.8000,  1.8000, -0.3000])


### 保存输出解读：奖励、价值和优势不是同一个数

以下解析于 **2026-10-10** 根据当前已保存输出补充，没有重新训练或替换输出。这里的数字是本次快照；以后重新运行时，应与新的输出对照。

| 经历 | 两人实际动作 | 团队奖励 / Critic 目标 | 更新前价值 | Actor 优势 |
|---|---|---:|---:|---:|
| 第 1 条 | `[0, 0]` | 2.0 | 0.2 | `2.0 - 0.2 = 1.8` |
| 第 2 条 | `[1, 1]` | 2.0 | 0.2 | `2.0 - 0.2 = 1.8` |
| 第 3 条 | `[0, 1]` | 0.0 | 0.3 | `0.0 - 0.3 = -0.3` |

**奖励**是教学规则给出的结果；**价值**是 Critic 对结果的预估；**优势**是实际结果相对旧预估的差。前两条比预期好，给实际动作正评分；第三条比预期差，给实际动作负评分。

三条经历都按一步真正终止处理，因此 Critic 的目标就是奖励 `[2, 2, 0]`，不再加未来价值。两人收到同一份团队评分，不把奖励 2 计算成 4，也不能凭第三条的共同负评分判断究竟是谁应负责。

`targets`、`old_values`、`advantages` 都已固定。Actor 更新时使用旧评分，不能让正在变化的 Critic 把这份评分一起改掉。

## 6. 每个 Actor 对应自己实际执行的动作

`log_prob_0` 必须来自智能体 0 当时执行的 `actions_0`，
另一名智能体同理。不能拿队友动作或新采样动作替换。

`actor_loss` 对两个角色的损失取平均，并对三条经历取平均。
这只是损失聚合方式，不是把环境奖励翻倍。
`critic_loss` 拟合固定团队回报，`0.5` 是它在联合损失中的权重。
本例用未标准化的原始评分，只更新一次，没有实现 PPO 裁剪。

In [6]:
log_prob_0 = dist_0.log_prob(actions_0)
log_prob_1 = dist_1.log_prob(actions_1)
loss_0 = -(log_prob_0 * advantages).mean()
loss_1 = -(log_prob_1 * advantages).mean()
actor_loss = (loss_0 + loss_1) / 2
critic_loss = ((value - targets) ** 2).mean()
total_loss = actor_loss + 0.5 * critic_loss

# None 表示该损失没有连接到这组参数，不是参数永久被冻结。
actor_to_critic = torch.autograd.grad(
    actor_loss, tuple(critic.parameters()),
    allow_unused=True, retain_graph=True,
)
critic_to_actors = torch.autograd.grad(
    critic_loss,
    tuple(actor_0.parameters()) + tuple(actor_1.parameters()),
    allow_unused=True, retain_graph=True,
)
print("Actor loss:", actor_loss.item())
print("Critic loss:", critic_loss.item())
print("总损失:", total_loss.item())
print("Actor loss 到 Critic 的梯度:", actor_to_critic)
print("Critic loss 到两个 Actor 的梯度:", critic_to_actors)

Actor loss: 0.7624619007110596
Critic loss: 2.18999981880188
总损失: 1.8574618101119995
Actor loss 到 Critic 的梯度: (None,)
Critic loss 到两个 Actor 的梯度: (None, None)


### 保存输出解读：三个损失如何对应上面的数字

Critic 的均方误差可以直接手算：

```text
MSE = ((0.2 - 2)^2 + (0.2 - 2)^2 + (0.3 - 0)^2) / 3
    = (3.24 + 3.24 + 0.09) / 3
    = 2.19
```

输出的 `2.1899998188` 是浮点表示下的同一结果。两个 Actor 初始实际动作概率都是 0.5，`log(0.5)` 约为 -0.693147；优势均值为 `(1.8 + 1.8 - 0.3) / 3 = 1.1`，所以每个 Actor 的损失约为 `-log(0.5) × 1.1 = 0.762462`。两份 Actor 损失取平均后仍为这个数。

总损失为 `0.762462 + 0.5 × 2.19 = 1.857462`。系数 0.5 调整 Critic 损失的权重，不是成功概率，也没有把团队奖励减半。Actor loss 的绝对大小不是合作成功率。

`Actor loss → Critic` 的梯度为 `(None,)`，因为优势已经脱离 Critic 的计算图；`Critic loss → Actor` 为 `(None, None)`，因为本例 Critic 不读取 Actor 的输出。**这不代表三个网络不学习**：各自的损失仍连接各自的参数，下一单元的联合反传会更新它们。

## 7. 执行一次真实参数更新

`zero_grad` 清空上一次梯度，`backward` 计算本次梯度，
`step` 才修改参数。先保存旧参数，后面直接核对有没有变化。
一个小批次的一次更新不是已经学会合作，也不保证每条样本的动作概率
都朝独立看这条样本时的方向变化。

In [7]:
models = {"actor_0": actor_0, "actor_1": actor_1, "critic": critic}
before_update = {
    name: model.weight.detach().clone() for name, model in models.items()
}
optimizer.zero_grad()
total_loss.backward()
gradients = {
    name: model.weight.grad.detach().clone() for name, model in models.items()
}
optimizer.step()

with torch.no_grad():
    after_values = critic(joint_features).squeeze(-1)
    after_probs_0 = Categorical(logits=actor_0(obs_0)).probs
    after_probs_1 = Categorical(logits=actor_1(obs_1)).probs
    after_critic_loss = ((after_values - targets) ** 2).mean()
print("更新后 Actor 0 概率:\n", after_probs_0)
print("更新后 Actor 1 概率:\n", after_probs_1)
print("更新后 Critic 预测:", after_values)
print("固定目标 MSE:", critic_loss.item(), "->", after_critic_loss.item())
for name in models:
    changed = not torch.equal(before_update[name], models[name].weight)
    print(name, "参数确实改变:", changed)

更新后 Actor 0 概率:
 tensor([[0.5031, 0.4969],
        [0.4956, 0.5044],
        [0.4987, 0.5012]])
更新后 Actor 1 概率:
 tensor([[0.5044, 0.4956],
        [0.4963, 0.5037],
        [0.5044, 0.4956]])
更新后 Critic 预测: tensor([0.2500, 0.2550, 0.3750])
固定目标 MSE: 2.18999981880188 -> 2.082716703414917
actor_0 参数确实改变: True
actor_1 参数确实改变: True
critic 参数确实改变: True


### 保存输出解读：平均误差下降，不等于每条经历都改善

| 经历 | 固定目标 | 更新前预测 | 更新后预测 | 单条平方误差的变化 |
|---|---:|---:|---:|---|
| 第 1 条 | 2 | 0.200 | 0.250 | `3.240000 → 3.062500` |
| 第 2 条 | 2 | 0.200 | 0.255 | `3.240000 → 3.045025` |
| 第 3 条 | 0 | 0.300 | 0.375 | `0.090000 → 0.140625` |

更新后的平均误差是 `(3.062500 + 3.045025 + 0.140625) / 3 = 2.0827167`，低于原来的 2.19。**第三条却变差了**：目标是 0，预测反而从 0.3 升到 0.375。

原因是三条经历共用一个线性 Critic 的参数。一次更新合并了整批样本的梯度，不是给每条经历单独存一个可独立修改的值；前两条推动参数上调的效果也会影响第三条。优化的对象是批次平均损失，不能要求每个样本在每一步都更接近目标。

Actor 的变化也要看**实际动作对应的那一列**：第 1 条两人动作 0 的概率分别约为 0.5031、0.5044；第 2 条两人动作 1 的概率约为 0.5044、0.5037；第 3 条 Actor 0 的动作 0 概率约为 0.4987、Actor 1 的动作 1 概率约为 0.4956。对照初始 0.5，本次正评分动作提高、负评分动作降低。

这些是一次小幅更新的结果。参数变化和平均误差下降说明更新链条确实发生了，还不能说明已学会稳定合作。

## 8. 控制变量：只改队友的观测

固定全部网络参数和 `obs_0`，只将 `obs_1` 换成另一份模拟输入。
Actor 0 没有读队友输入，所以它的输出不变。
Critic 读了联合特征，在当前手工初值及这一次更新后的权重下，
它的输出会改变。这里只验证输入依赖，不声称新预测准确。

In [8]:
other_obs_1 = obs_1 + 5.0
with torch.no_grad():
    same_local_probs = Categorical(logits=actor_0(obs_0)).probs
    other_features = torch.cat([obs_0, other_obs_1], dim=-1)
    other_values = critic(other_features).squeeze(-1)
print("Actor 0 概率是否完全相同:",
      torch.equal(after_probs_0, same_local_probs))
print("原联合输入的价值:", after_values)
print("改队友输入后的价值:", other_values)

Actor 0 概率是否完全相同: True
原联合输入的价值: tensor([0.2500, 0.2550, 0.3750])
改队友输入后的价值: tensor([1.5250, 1.5300, 1.6500])


## 9. 分散执行：没有 Critic 也能选择动作

`choose_action` 只接收一个 Actor 和它自己的局部观测。
此处用最大概率动作做确定性的接口演示，不代表所有执行都必须贪心。
前向钩子只用于数 Critic 被调用几次，不改变任何网络输出。
实际调用两个 Actor 时计数应为 0；额外用一次 Critic 前向确认计数器有效。
执行过程中使用 `no_grad`，不更新网络参数。

In [9]:
@torch.no_grad()
def choose_action(actor, local_obs):
    distribution = Categorical(logits=actor(local_obs))
    return distribution.probs.argmax(dim=-1)

before_execution = {
    name: model.weight.detach().clone() for name, model in models.items()
}
critic_calls = []
handle = critic.register_forward_hook(
    lambda module, inputs, output: critic_calls.append("called")
)
try:
    executed_0 = choose_action(actor_0, obs_0)
    executed_1 = choose_action(actor_1, obs_1)
    execution_critic_calls = len(critic_calls)
    # 校准计数器：直接调用一次 Critic，必须能够记录到。
    with torch.no_grad():
        critic(joint_features)
    hook_calibration_calls = len(critic_calls)
finally:
    handle.remove()

print("执行时 Actor 0 的动作:", executed_0)
print("执行时 Actor 1 的动作:", executed_1)
print("执行阶段 Critic 调用次数:", execution_critic_calls)
print("计数器校准结果:", hook_calibration_calls)

执行时 Actor 0 的动作: tensor([0, 1, 1])
执行时 Actor 1 的动作: tensor([0, 1, 0])
执行阶段 Critic 调用次数: 0
计数器校准结果: 1


### 保存输出解读：控制变量和执行计数验证了什么

上一节只把队友观测加了 5，Actor 0 的输入保持原样，因此它的动作分布完全不变；Critic 的联合输入改变，预测由 `[0.250, 0.255, 0.375]` 变为 `[1.525, 1.530, 1.650]`。这验证了**谁依赖哪些输入**，不是证明修改后的价值预测准确；没有新奖励为这些大数提供正确性依据。

执行阶段使用 `argmax`，Actor 0 输出 `[0, 1, 1]`，Actor 1 输出 `[0, 1, 0]`。每个位置对应原来同一行观测。第三条的局部贪心动作仍不相同，不能把前两条动作相同解释成整个合作任务已经训练成功。

`执行阶段 Critic 调用次数 = 0` 表示这次选择动作只用了局部 Actor；随后人为调用 Critic 一次，计数器变为 1，说明零次调用不是计数器失效。这里没有重新与环境交互，也没有计算新的成功率。

后面的 **32/32 项检查**覆盖样本配对、目标固定、梯度连接、真实参数更新与分散执行接口。它们检查的是这份数据流练习是否按设计工作，不是 32 次环境评估。

## 10. 自动核对

以下检查针对形状、固定参照、梯度、真实更新及执行信息边界。
通过表示这个教学数据流可运行，不表示完整 MAPPO 已复现。

In [10]:
checks = {}

def check(name, condition):
    passed = bool(condition)
    checks[name] = passed
    assert passed, name

check("local observation shapes", obs_0.shape == obs_1.shape == (3, 2))
check("joint feature shape", joint_features.shape == (3, 4))
check("joint feature row pairing",
      torch.equal(joint_features[0], torch.tensor([1., 0., 0., 1.])))
check("shared rewards not doubled", torch.equal(rewards, torch.tensor([2., 2., 0.])))
check("action index dtype", actions_0.dtype == actions_1.dtype == torch.long)
check("actor output shapes", dist_0.probs.shape == dist_1.probs.shape == (3, 2))
check("initial uniform actors",
      torch.allclose(dist_0.probs, torch.full((3, 2), 0.5))
      and torch.allclose(dist_1.probs, torch.full((3, 2), 0.5)))
check("value batch dimension", value.shape == (3,))
check("singleton batch squeeze is safe",
      critic(joint_features[:1]).squeeze(-1).shape == (1,))
check("old value numbers",
      torch.allclose(old_values, torch.tensor([0.2, 0.2, 0.3])))
check("fixed raw advantages",
      torch.allclose(advantages, torch.tensor([1.8, 1.8, -0.3])))
check("fixed targets have no gradient", not targets.requires_grad)
check("advantages have no gradient", not advantages.requires_grad)
check("current critic value retains gradient", value.requires_grad)
check("actual action log probabilities",
      log_prob_0.shape == log_prob_1.shape == (3,))
check("actor loss does not train critic", all(g is None for g in actor_to_critic))
check("critic loss does not train actors", all(g is None for g in critic_to_actors))
check("critic MSE matches hand calculation",
      math.isclose(critic_loss.item(), 2.19, abs_tol=1e-6))
for name, model in models.items():
    check(name + " nonzero gradient", gradients[name].abs().sum() > 0)
    check(name + " parameter changed",
          not torch.equal(before_update[name], model.weight))
check("fixed target MSE decreased", after_critic_loss < critic_loss.detach())
check("changing partner input leaves local actor unchanged",
      torch.equal(after_probs_0, same_local_probs))
check("critic responds to changed joint features",
      not torch.allclose(after_values, other_values))
check("actor-only execution", execution_critic_calls == 0)
check("critic hook calibrated", hook_calibration_calls == 1)
check("execution produces one action per row",
      executed_0.shape == executed_1.shape == (3,))
check("execution does not change parameters",
      all(torch.equal(before_execution[name], model.weight)
          for name, model in models.items()))
check("targets unchanged after update and execution",
      torch.equal(targets, rewards))
print(f"通过 {sum(checks.values())}/{len(checks)} 项检查")

summary = {
    "checks_passed": sum(checks.values()),
    "checks_total": len(checks),
    "before_values": old_values.tolist(),
    "after_values": after_values.tolist(),
    "before_critic_loss": critic_loss.item(),
    "after_critic_loss": after_critic_loss.item(),
    "actor_0_probs": after_probs_0.tolist(),
    "actor_1_probs": after_probs_1.tolist(),
    "execution_critic_calls": execution_critic_calls,
}
print(summary)

通过 32/32 项检查
{'checks_passed': 32, 'checks_total': 32, 'before_values': [0.20000000298023224, 0.20000000298023224, 0.30000001192092896], 'after_values': [0.25, 0.2549999952316284, 0.375], 'before_critic_loss': 2.18999981880188, 'after_critic_loss': 2.082716703414917, 'actor_0_probs': [[0.503125011920929, 0.4968750476837158], [0.4956251084804535, 0.5043749213218689], [0.4987499713897705, 0.5012499690055847]], 'actor_1_probs': [[0.5043749213218689, 0.4956251084804535], [0.49625006318092346, 0.5037499070167542], [0.5043749213218689, 0.4956251084804535]], 'execution_critic_calls': 0}


## 与 PPO、MAPPO 的连接

CTDE 是训练/执行信息约束范式，不等于某一个算法，也不要求一个中央机器。
本例选择集中式状态价值 Critic 和分开的局部 Actor，只演示一种数据流。
集中式 Critic 不一定只有一个，也不一定使用拼接观测作为输入。

接到 Day5 PPO 时，仍需近期联合 rollout、各角色实际动作的旧概率、
固定优势与价值目标、有限多轮裁剪更新和冻结评估。
使用集中式价值输入的多智能体 PPO 对应 MAPPO 的核心结构；
Actor 与 Critic 都只用各自局部输入的 PPO 对应 IPPO。
本练习没有实现上述完整循环，不能据此声称合作训练成功。

Actor 执行所需的输入必须在执行时真实可得。集中式 Critic 的训练信息
不应直接成为执行时缺失的 Actor 输入。执行时可用的通信或历史信息
可以成为策略输入，但不能依靠训练时才存在的特权信息。

概念参考：Lowe 等，Multi-Agent Actor-Critic，第 1、4 节；
Yu 等，The Surprising Effectiveness of PPO，第 3.2、5.2 节。